### Importing necessary libraries:

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

### Reading CDC Diabetes Health Indicators Dataset

In [2]:
DATA_DIR = "../data/raw/"
DATASET_PATH = os.path.join(DATA_DIR, "diabetes-menhlthl-phyhlth_preprocessing.csv")

df = pd.read_csv(DATASET_PATH)
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

print(df.columns)
print(df.head())
print(df.info())
print(df.describe())

Dataset: 253680 rows, 22 columns
Index(['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')
   Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1              0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2              0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3              0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4              0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...   

### Mental / Physical Health Targets 
We aim to predict Diabetes Risk, Mental Health Risk, Physical Health Risk all from this dataset.\
Flagging > 10 days (11-30 days) of poor health as Risky. 

In [3]:
df['Risky_Mental_Health'] = (df['MentHlth'] > 10).astype(int)
df['Risky_Physical_Health'] = (df['PhysHlth'] > 10).astype(int)

### Separate Features (X) and 3 Targets (y)
We strip all three targets out to leave only the lifestyle / biological features

In [4]:
X = df.drop(columns=['Diabetes_binary', 'Risky_Mental_Health', 'Risky_Physical_Health'])

y_diabetes = df['Diabetes_binary']
y_mental = df['Risky_Mental_Health']
y_physical = df['Risky_Physical_Health']

### Train/Test Split

In [5]:
print("Splitting data into Train and Test sets...")
X_train, X_test, y_train_diabetes, y_test_diabetes, y_train_mental, y_test_mental, y_train_physical, y_test_physical = train_test_split(
    X, y_diabetes, y_mental, y_physical, test_size=0.2, random_state=42
)

Splitting data into Train and Test sets...


### Feature Scaling

In [6]:
continuous_cols = ['BMI', 'Age', 'Education', 'Income', 'GenHlth']

scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

### Handling Class Imbalance (SMOTE for Diabetes)

In [7]:
print("Applying SMOTE to balance the Diabetes training data...")
smote = SMOTE(random_state=42)
X_train_diabetes_balanced, y_train_diabetes_balanced = smote.fit_resample(X_train, y_train_diabetes)

Applying SMOTE to balance the Diabetes training data...


### Summary Check

In [8]:
print("\n=== PIPELINE COMPLETE ===")
print(f"Features mapped: {X_train.shape[1]}")
print("\nTarget 1: Diabetes Risk (Training Data Balanced via SMOTE)")
print(y_train_diabetes_balanced.value_counts(normalize=True).to_dict())
print("\nTarget 2: Mental Health Risk (Original Distribution)")
print(y_train_mental.value_counts(normalize=True).to_dict())
print("\nTarget 3: Physical Health Risk (Original Distribution)")
print(y_train_physical.value_counts(normalize=True).to_dict())


=== PIPELINE COMPLETE ===
Features mapped: 21

Target 1: Diabetes Risk (Training Data Balanced via SMOTE)
{0.0: 0.5, 1.0: 0.5}

Target 2: Mental Health Risk (Original Distribution)
{0: 0.9011057237464523, 1: 0.09889427625354778}

Target 3: Physical Health Risk (Original Distribution)
{0: 0.8643024676758121, 1: 0.13569753232418796}
